# Health QA Colab Seq2Seq LoRA Fine-Tuning

This notebook runs a GPU-only experiment branch for stronger seq2seq fine-tuning. It does not replace the stable public-best submission unless the generated file beats `0.602576` on Zindi.

Recommended first run: `configs/colab_afriteva_v2_lora_full.yaml`.

Fallback run: `configs/colab_mt5_base_lora_full.yaml`.

In [ ]:
# Runtime check. Use Runtime > Change runtime type > T4/A100 GPU before continuing.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('Switch this Colab runtime to GPU before running the fine-tune.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/elvispreakerebi/health-qa-summative.git'
BRANCH = 'colab-llm-finetune'
REPO_DIR = '/content/health-qa-summative'
DRIVE_DATA_DIR = '/content/drive/MyDrive/mlt1-summative-datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/health-qa-summative-outputs'

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!mkdir -p data/raw {DRIVE_OUTPUT_DIR}
!cp {DRIVE_DATA_DIR}/Train.csv data/raw/Train.csv
!cp {DRIVE_DATA_DIR}/Val.csv data/raw/Val.csv
!cp {DRIVE_DATA_DIR}/Test.csv data/raw/Test.csv

In [ ]:
# Colab dependency setup. Restart runtime only if Colab asks after installation.
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e .

In [ ]:
# Confirm schema and row counts before training.
!python -m health_qa.cli inspect-data --config configs/colab_afriteva_v2_lora_full.yaml

In [ ]:
# Choose exactly one config per run.
CONFIG = 'configs/colab_afriteva_v2_lora_full.yaml'
# CONFIG = 'configs/colab_mt5_base_lora_full.yaml'

RUN_NAME = CONFIG.split('/')[-1].replace('.yaml', '')
OUTPUT_DIR = f'outputs/{RUN_NAME}'
print(CONFIG, OUTPUT_DIR)

In [ ]:
# Fine-tune and generate validation/test predictions.
# This is the long-running cell. Keep the browser/session active until it prints the submission path.
!python -m health_qa.cli train-generate --config {CONFIG} --output-dir {OUTPUT_DIR}

In [ ]:
import pandas as pd
from health_qa.submission import validate_submission

metrics = pd.read_csv(f'{OUTPUT_DIR}/metrics.csv')
submission = pd.read_csv(f'{OUTPUT_DIR}/submission.csv')
validate_submission(submission)
display(metrics)
display(submission.head())
print('Submission rows:', len(submission))
print('Submission path:', f'{OUTPUT_DIR}/submission.csv')

In [ ]:
# Persist artifacts to Drive so they survive Colab disconnects.
!mkdir -p {DRIVE_OUTPUT_DIR}/{RUN_NAME}
!cp {OUTPUT_DIR}/metrics.csv {DRIVE_OUTPUT_DIR}/{RUN_NAME}/metrics.csv
!cp {OUTPUT_DIR}/validation_predictions.csv {DRIVE_OUTPUT_DIR}/{RUN_NAME}/validation_predictions.csv
!cp {OUTPUT_DIR}/submission.csv {DRIVE_OUTPUT_DIR}/{RUN_NAME}/submission.csv
!cp -r {OUTPUT_DIR}/final_model {DRIVE_OUTPUT_DIR}/{RUN_NAME}/final_model
print('Saved to:', f'{DRIVE_OUTPUT_DIR}/{RUN_NAME}')